# 4 — Training on Google Colab

Runs the same `nmt.training.train` entry point used locally, on a GPU runtime.

**Before running:** Runtime → Change runtime type → **T4 GPU** (or better).

The primary model (`bpe_scratch`) sees 271k pairs → **543k directional
examples**, about 12M source+target tokens per epoch, through a 37.6M-parameter
model.

| Stage | T4 estimate |
|---|---|
| Download + build corpus | ~5 min |
| One training epoch | ~8–15 min |
| Full run (early stopping usually fires around epoch 10–14) | ~2–3 h |
| Test-set evaluation (beam 4, both directions) | ~10–15 min |

For reference, the same epoch takes **~3.4 hours** on an M-series MacBook GPU
(measured: ~1,030 tokens/s), which is why training runs here rather than
locally.

### Train in priority order

Colab free sessions are time-limited, so train in the order that protects the
most marks if you run out:

1. **`bpe_scratch`** — the primary model. Everything downstream needs it: the
   error analysis, the app, most of the report.
2. **`word_random` + `word_muse`** — train these two *as a pair* or not at all.
   `word_muse` alone proves nothing; `word_random` is the control that makes it
   interpretable.
3. **`lstm_baseline`** — the architecture comparison. Slowest per epoch,
   because its decoder has an explicit Python time loop.


In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Clone the repository
#    Replace the URL with your own fork if you have pushed changes.
REPO = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"

import os
if not os.path.exists("Final_Project"):
    !git clone $REPO Final_Project
%cd Final_Project
!ls

In [ ]:
# 3. Install the dependencies Colab does not already have
!pip install -q sentencepiece sacrebleu

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())

In [ ]:
# 4. Mount Drive so checkpoints survive a disconnect
from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/aig230_nmt/checkpoints"
RESULTS_DIR = "/content/drive/MyDrive/aig230_nmt/results"
!mkdir -p "$CHECKPOINT_DIR" "$RESULTS_DIR"

# Symlink so the code writes straight to Drive without any config change.
!rm -rf artifacts/checkpoints artifacts/results
!mkdir -p artifacts
!ln -s "$CHECKPOINT_DIR" artifacts/checkpoints
!ln -s "$RESULTS_DIR" artifacts/results
!ls -la artifacts/

In [ ]:
# 5. Build the corpus (~5 min, downloads 172 MB)
!PYTHONPATH=src python -m nmt.data.build

In [ ]:
# 6. Verify the pipeline before spending GPU hours on it
!PYTHONPATH=src python -m nmt.training.train --config configs/smoke.yaml

## Train the primary model

If the session disconnects, re-run the cells above and then add
`--resume artifacts/checkpoints/bpe_scratch/last.pt` — checkpoints carry the
optimiser and scheduler state, so training continues rather than restarting the
warmup.

In [ ]:
!PYTHONPATH=src python -m nmt.training.train --config configs/bpe_scratch.yaml

## The other three systems

`word_muse` needs the MUSE vectors (~1.3 GB); skip that cell if you only want
the architecture comparison.

In [ ]:
# Pre-trained cross-lingual vectors (only needed for word_muse)
!mkdir -p data/raw/embeddings
!wget -q --show-progress -O data/raw/embeddings/wiki.multi.en.vec https://dl.fbaipublicfiles.com/arrival/vectors/wiki.multi.en.vec
!wget -q --show-progress -O data/raw/embeddings/wiki.multi.es.vec https://dl.fbaipublicfiles.com/arrival/vectors/wiki.multi.es.vec
!PYTHONPATH=src python -m nmt.data.embeddings

In [ ]:
for config in ["word_random", "word_muse", "lstm_baseline"]:
    print(f"\n{'=' * 60}\n{config}\n{'=' * 60}")
    !PYTHONPATH=src python -m nmt.training.train --config configs/{config}.yaml

## Evaluate everything

In [ ]:
import os

for run in ["bpe_scratch", "word_random", "word_muse", "lstm_baseline"]:
    checkpoint = f"artifacts/checkpoints/{run}/best_bleu.pt"
    if os.path.exists(checkpoint):
        print(f"\n=== {run} ===")
        !PYTHONPATH=src python -m nmt.evaluation.evaluate --checkpoint {checkpoint}

In [ ]:
# Regenerate the figures and the report tables from the fresh results
!PYTHONPATH=src python -m nmt.viz.make_figures
!PYTHONPATH=src python reports/build_report_data.py

import json
from pathlib import Path

for path in sorted(Path("artifacts/results").glob("*/evaluation_test.json")):
    report = json.loads(path.read_text())
    print(f"{path.parent.name:16s} "
          f"EN->ES {report['summary']['bleu']['en-es']:6.2f}   "
          f"ES->EN {report['summary']['bleu']['es-en']:6.2f}")

In [ ]:
# Package the artefacts the report needs so they can be downloaded in one file
!zip -qr results.zip artifacts/results reports/figures reports/final_report/generated
from google.colab import files
files.download("results.zip")